# Expanded MD&A experiments — Colab GPU only

This run expands the supplied MD&A sample toward the 111,189-row fixed text manifest. It adds strict Item 7/Item 2 text, document-max embedding pooling, prior same-issuer/same-form text similarity and embedding changes, word/character TF-IDF, and an XGBoost pairwise ranker grouped by filing year and form. All model fitting and sentence embedding require CUDA; there is no CPU fitting fallback.

The headline results remain year/form matched 1:1. The 2024Q4–2025Q3 period has already been inspected, so it remains development data. A stable 20% CIK holdout is excluded from model selection and fitting and reported separately as an exploratory issuer-disjoint check; because the dates have already been examined in aggregate, it is not a pristine test. Scores are academic amendment-proxy results, not confirmed-restatement or natural-population estimates.

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU and rerun."
print("CUDA:",torch.version.cuda,"GPU:",torch.cuda.get_device_name(0))
subprocess.run(["nvidia-smi"],check=True)

In [ ]:
%pip -q uninstall -y xgboost xgboost-cu12
%pip -q install "xgboost-cu12>=3.2,<4" "scikit-learn>=1.5,<2" "sentence-transformers>=3,<6" "pandas>=2.2,<3" "numpy>=1.26,<3" "pyarrow>=16,<25" "scipy>=1.12,<2" "joblib>=1.3,<2"

If Colab requests a runtime restart after installing packages, restart, select GPU again, and run all cells from the top.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
ROOT=Path("/content/drive/MyDrive/financial-risk-detector")
EXPANDED_TEXT_PATH=ROOT/"data/colab_mda_text_expanded.csv.gz"
SAMPLED_TEXT_PATH=ROOT/"data/colab_mda_text_case_control.csv.gz"
TEXT_PATH=EXPANDED_TEXT_PATH if EXPANDED_TEXT_PATH.is_file() else SAMPLED_TEXT_PATH
PROXY_PATH=ROOT/"data/colab_amendment_proxy.csv.gz"
assert TEXT_PATH.is_file(), "Upload the existing case-control text CSV, or finish the companion SEC section-preparation notebook."
assert PROXY_PATH.is_file(), f"Missing current proxy labels: {PROXY_PATH}"
print("Text input:",TEXT_PATH,"rows are read by the training pipeline; expanded sections are preferred when present.")

In [ ]:
from pathlib import Path
import sys, importlib, importlib.util
MODULES=Path("/content/mda_advanced_gpu_modules"); MODULES.mkdir(parents=True,exist_ok=True)
SOURCES={'research_v5_core.py': '"""Point-in-time features and fixed study design; no model fitting or networking."""\nimport hashlib\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nTARGET = \'y_amendment_proxy\'\nSEEDS = (42, 143, 244)\nFOLDS = ((2018, 2020), (2020, 2022))\nRATIOS = (\'accruals_to_assets\', \'net_margin\', \'gross_margin\', \'debt_to_assets\',\n          \'cash_to_assets\', \'working_capital_to_assets\', \'receivables_to_revenue\',\n          \'inventory_to_revenue\', \'operating_cashflow_to_assets\', \'revenue_to_assets\',\n          \'liabilities_to_assets\', \'retained_earnings_to_assets\', \'filing_delay_days\')\nEXCLUDED = {\'adsh\', \'cik\', \'period\', \'period_date\', \'filed\', \'filed_date\', \'filed_year\',\n            \'sic\', \'afs\', \'wksi\', \'days_to_amendment\', TARGET, \'feature_schema_version\',\n            \'label_observation_cutoff\'}\n\n\ndef file_hash(path):\n    result = hashlib.sha256()\n    with open(path, \'rb\') as handle:\n        for chunk in iter(lambda: handle.read(2**20), b\'\'):\n            result.update(chunk)\n    return result.hexdigest()\n\n\ndef load_data(path):\n    df = pd.read_csv(path, dtype={\'adsh\':str, \'cik\':str}, low_memory=False)\n    required = EXCLUDED | {\'form\', \'fp\', \'sic2\', \'prior_amendment_count\'}\n    if required - set(df):\n        raise ValueError(f\'Missing columns: {sorted(required-set(df))}\')\n    if not df.feature_schema_version.eq(4).all():\n        raise ValueError(\'Use the prepared schema-v4 CSV, not an earlier export.\')\n    df[\'filed_date\'] = pd.to_datetime(df.filed_date, errors=\'raise\')\n    df[\'period_date\'] = pd.to_datetime(df.period_date, errors=\'raise\')\n    df[\'cik\'] = df.cik.astype(str).str.lstrip(\'0\').replace(\'\', \'0\')\n    if not df.adsh.is_unique or df[TARGET].isna().any() or not set(df[TARGET]) <= {0,1}:\n        raise ValueError(\'Invalid labels or duplicate accessions.\')\n    cutoff = pd.to_datetime(df.label_observation_cutoff)\n    if cutoff.nunique() != 1 or cutoff.iloc[0] < pd.Timestamp(\'2026-09-30\'):\n        raise ValueError(\'Require label observation through 2026-09-30.\')\n    if not ((cutoff-df.filed_date).dt.days >= 365).all():\n        raise ValueError(\'An outcome has less than 365 days of follow-up.\')\n    if not set(df.form) <= {\'10-K\',\'10-Q\'}:\n        raise ValueError(\'Predictors must come from original 10-K/10-Q filings.\')\n    df[\'sic2\'] = df.sic2.fillna(\'unknown\').astype(str)\n    return df.reset_index(drop=True)\n\n\ndef balanced(part, seed=42):\n    pos, neg = part[part[TARGET].eq(1)], part[part[TARGET].eq(0)]\n    if pos.empty:\n        raise ValueError(\'Cohort has zero positives.\')\n    controls = []\n    for (year, form), group in pos.groupby([\'filed_year\',\'form\']):\n        pool = neg[neg.filed_year.eq(year) & neg.form.eq(form)]\n        if len(pool) < len(group):\n            raise ValueError(f\'Insufficient controls: {year}, {form}\')\n        controls.append(pool.sample(len(group), replace=False, random_state=seed))\n    return pd.concat([pos, *controls]).sample(frac=1, random_state=seed)\n\n\ndef date_part(df, start, end):\n    return df[df.filed_date.between(pd.Timestamp(start),pd.Timestamp(end))]\n\n\ndef training_pool(df, fit_year, window=8):\n    return date_part(df, f\'{max(2009,fit_year-window+1)}-01-01\', f\'{fit_year}-12-31\')\n\n\ndef add_mature_history(df):\n    """A prior label is usable only AFTER its complete 365-day outcome window."""\n    result = pd.DataFrame(index=df.index)\n    for group_cols, prefix in [([\'cik\'],\'issuer\'), ([\'cik\',\'form\'],\'issuer_form\')]:\n        count = np.zeros(len(df)); positive = np.zeros(len(df))\n        for _, group in df.groupby(group_cols, sort=False):\n            history = group.sort_values(\'filed_date\')\n            maturity = (history.filed_date + pd.Timedelta(days=365)).to_numpy(\'datetime64[D]\')\n            at = np.searchsorted(maturity, group.filed_date.to_numpy(\'datetime64[D]\'), side=\'left\')\n            prefix_sum = np.r_[0, history[TARGET].to_numpy().cumsum()]\n            count[group.index] = at\n            positive[group.index] = prefix_sum[at]\n        result[f\'{prefix}_mature_count\'] = count\n        result[f\'{prefix}_mature_positive_count\'] = positive\n        # Fixed smoothing constant; never estimated from later outcome labels.\n        result[f\'{prefix}_mature_rate\'] = (positive + 0.4) / (count + 20)\n    return pd.concat([df, result], axis=1)\n\n\ndef add_financial_history(df):\n    columns = [c for c in RATIOS if c in df]\n    # One aggregate per issuer/form/day prevents same-day information leakage.\n    keys = [\'cik\',\'form\',\'filed_date\']\n    days = df.groupby(keys, as_index=False)[columns].mean().sort_values(keys)\n    groups = days.groupby([\'cik\',\'form\'], sort=False)\n    # Compute all ratios together; per-issuer Python callbacks are very slow\n    # across hundreds of thousands of filings on a standard Colab runtime.\n    past = groups[columns].shift()\n    rolling = past.groupby([days.cik,days.form],sort=False).rolling(3,min_periods=1)\n    means = rolling.mean().reset_index(level=[0,1],drop=True)\n    stds = rolling.std().reset_index(level=[0,1],drop=True)\n    derived = []\n    for col in columns:\n        days[f\'past3_mean_{col}\'] = means[col]\n        days[f\'past3_std_{col}\'] = stds[col]\n        derived += [f\'past3_mean_{col}\', f\'past3_std_{col}\']\n    out = df.merge(days[keys+derived], on=keys, how=\'left\', validate=\'many_to_one\')\n    extra = {}\n    for col in columns:\n        extra[f\'deviation_past3_{col}\'] = out[col]-out[f\'past3_mean_{col}\']\n    out = pd.concat([out, pd.DataFrame(extra,index=out.index)], axis=1)\n    # Calendar-year aggregates come solely from the PREVIOUS filing year.\n    for col in columns:\n        stats = df.groupby([\'filed_year\',\'form\',\'sic2\'])[col].agg([\'median\',\'count\']).reset_index()\n        stats[\'filed_year\'] += 1\n        stats.loc[stats[\'count\'] < 20, \'median\'] = np.nan\n        name = f\'prior_year_peer_median_{col}\'\n        stats = stats.rename(columns={\'median\':name})\n        out = out.merge(stats[[\'filed_year\',\'form\',\'sic2\',name]],\n                        on=[\'filed_year\',\'form\',\'sic2\'], how=\'left\', validate=\'many_to_one\')\n        out[f\'peer_deviation_{col}\'] = out[col]-out[name]\n    out[\'loss_with_positive_cashflow\'] = ((out.net_income_to_assets < 0) &\n                                           (out.operating_cashflow_to_assets > 0)).astype(\'int8\')\n    out[\'profit_with_negative_cashflow\'] = ((out.net_income_to_assets > 0) &\n                                            (out.operating_cashflow_to_assets < 0)).astype(\'int8\')\n    out[\'negative_equity\'] = out.xbrl_StockholdersEquity.lt(0).astype(\'int8\')\n    facts = [c for c in df if c.startswith(\'xbrl_\')]\n    out[\'financial_fact_missing_fraction\'] = out[facts].isna().mean(axis=1)\n    return out\n\n\ndef add_event_history(df, events):\n    """Public EDGAR index events, all strictly before the predictor date."""\n    out = pd.DataFrame(index=df.index)\n    events = events.copy()\n    events[\'filed_date\'] = pd.to_datetime(events.filed_date)\n    events[\'cik\'] = events.cik.astype(str).str.lstrip(\'0\').replace(\'\',\'0\')\n    categories = {\'8k\':[\'8-K\',\'8-K/A\'], \'late\':[\'NT 10-K\',\'NT 10-Q\'],\n                  \'10ka\':[\'10-K/A\'], \'10qa\':[\'10-Q/A\']}\n    for name, forms in categories.items():\n        event_groups = {cik:np.sort(g.filed_date.to_numpy(\'datetime64[D]\'))\n                        for cik,g in events[events.form.isin(forms)].groupby(\'cik\')}\n        counts = {n:np.zeros(len(df), dtype=\'int32\') for n in (90,365,730)}\n        gap = np.full(len(df), np.nan)\n        for cik, rows in df.groupby(\'cik\',sort=False):\n            dates = event_groups.get(cik, np.array([],dtype=\'datetime64[D]\'))\n            query = rows.filed_date.to_numpy(\'datetime64[D]\')\n            end = np.searchsorted(dates,query,side=\'left\')\n            for span in counts:\n                counts[span][rows.index] = end-np.searchsorted(dates,query-np.timedelta64(span,\'D\'),side=\'left\')\n            valid = end > 0\n            gap[rows.index[valid]] = (query[valid]-dates[end[valid]-1]).astype(\'timedelta64[D]\').astype(int)\n        for span,values in counts.items():\n            out[f\'sec_{name}_count_{span}d\'] = values\n        out[f\'sec_{name}_days_since\'] = gap\n    return pd.concat([df,out],axis=1)\n\n\ndef threshold_choice(y, probability):\n    """Maximise the worst of precision/recall/F1/accuracy on CALIBRATION only."""\n    y = np.asarray(y, dtype=int); p = np.asarray(probability, dtype=float)\n    if len(y) == 0 or len(np.unique(y)) < 2 or not np.isfinite(p).all():\n        raise ValueError(\'Threshold selection requires finite scores and both classes.\')\n    order = np.argsort(-p, kind=\'stable\'); ys = y[order]; ps = p[order]\n    ends = np.r_[np.flatnonzero(ps[:-1] != ps[1:]),len(ps)-1]\n    tp = ys.cumsum()[ends]; fp = ends+1-tp\n    recall = tp/y.sum(); precision = tp/(ends+1)\n    f1 = 2*precision*recall/np.maximum(precision+recall,1e-12)\n    accuracy = (tp+(1-y).sum()-fp)/len(y)\n    floor = np.minimum.reduce([precision,recall,f1,accuracy])\n    i = int(np.argmax(floor))\n    return {\'threshold\':float(ps[ends[i]]), \'calibration_min_metric\':float(floor[i]),\n            \'calibration_all_metrics_ge_80\':bool(np.max(floor) >= .80),\n            \'calibration_all_metrics_ge_85\':bool(np.max(floor) >= .85)}\n\n\ndef text_manifest(df):\n    parts = []\n    for fit_year in (2018,2020,2021):\n        pool = training_pool(df,fit_year)\n        parts += [balanced(pool,seed) for seed in SEEDS]\n    for start,end in [(\'2020-01-01\',\'2020-12-31\'),(\'2022-01-01\',\'2022-12-31\'),\n                      (\'2023-01-01\',\'2023-09-30\'),(\'2024-10-01\',\'2025-09-30\')]:\n        parts.append(date_part(df,start,end))\n    return pd.concat(parts).drop_duplicates(\'adsh\')[[\'adsh\',\'cik\',\'form\',\'filed_date\']].sort_values(\'adsh\')\n', '17_train_mda_text_gpu.py': '"""GPU-only, selected-case-control MD&A experiment using current proxy labels.\n\nThis is an exploratory balanced-cohort comparison, not a natural-population\nevaluation. The source MD&A corpus was historically selected and 50/50 sampled.\nIts old `y` label is never loaded here; prepared `y_proxy` is the sole target.\n"""\nfrom pathlib import Path\nimport hashlib\nimport importlib.metadata\nimport json\nimport re\nimport sys\nimport numpy as np\nimport pandas as pd\nimport research_v5_core as core\n\n\ndef atomic_json(path,obj):\n    temp=Path(str(path)+\'.part\')\n    temp.write_text(json.dumps(obj,indent=2),encoding=\'utf-8\')\n    temp.replace(path)\n\n\ndef atomic_npy(path,values):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    temp=path.with_suffix(\'.part.npy\')\n    np.save(temp,np.asarray(values))\n    temp.replace(path)\n\n\ndef stable_signature(value):\n    return hashlib.sha256(json.dumps(value,sort_keys=True,default=str).encode()).hexdigest()\n\n\ndef issuer_holdout_mask(cik, fraction=.20, salt=\'mda-expanded-cik-v1\'):\n    """Stable issuer split; no company can cross model selection and holdout."""\n    if not 0 < fraction < 1:\n        raise ValueError(\'Issuer holdout fraction must be strictly between zero and one.\')\n    limit=int(fraction*(2**64))\n    return np.asarray([\n        int.from_bytes(hashlib.sha256(f\'{salt}:{str(value).lstrip("0") or "0"}\'.encode()).digest()[:8],\n                       \'big\') < limit for value in cik],dtype=bool)\n\n\ndef matched_cohort(part,seed):\n    rows=[]\n    part=part.copy()\n    part[\'filed_year\']=pd.to_datetime(part.filed_date).dt.year\n    for _,group in part.groupby([\'filed_year\',\'form\'],sort=True):\n        positive=group[group.y_amendment_proxy.eq(1)]\n        negative=group[group.y_amendment_proxy.eq(0)]\n        count=min(len(positive),len(negative))\n        if count:\n            rows.extend([positive.sample(count,random_state=seed),\n                         negative.sample(count,random_state=seed)])\n    if not rows:\n        raise ValueError(\'No positive/negative year-form strata can be matched.\')\n    return pd.concat(rows).sample(frac=1,random_state=seed).reset_index(drop=True)\n\n\ndef balanced_train(part,seed):\n    part=part.copy()\n    part[\'filed_year\']=pd.to_datetime(part.filed_date).dt.year\n    cohorts=[]\n    # Match within filing year and form so the model cannot gain from the\n    # selected corpus\' changing case/control composition across time or forms.\n    for _,group in part.groupby([\'filed_year\',\'form\'],sort=True):\n        positive=group[group.y_amendment_proxy.eq(1)]\n        negative=group[group.y_amendment_proxy.eq(0)]\n        count=min(len(positive),len(negative))\n        if count:\n            cohorts.extend([positive.sample(count,random_state=seed),\n                            negative.sample(count,random_state=seed)])\n    if not cohorts:\n        raise ValueError(\'No positive/negative year-form training strata can be matched.\')\n    return pd.concat(cohorts).sample(frac=1,random_state=seed).reset_index(drop=True)\n\n\ndef columns_for(train,with_tabular):\n    categories=[c for c in (\'form\',\'fp\',\'sic2\') if c in train]\n    # The legacy corpus includes a `y` column whose labels disagree with the\n    # current accession-joined proxy. Never allow it, or common label aliases,\n    # to enter predictors.\n    excluded=core.EXCLUDED|set(categories)|{\n        \'mda_text\',\'y_proxy\',\'y\',\'legacy_y\',\'label\',\'label_source\',\n        \'adsh_amendment\',\'filed_amendment\',\'amendment_url\'}\n    numeric=[c for c in train if c not in excluded and\n             pd.api.types.is_numeric_dtype(train[c]) and\n             (with_tabular or c.startswith((\'text_emb_\',\'text_style_\')) or c==\'mda_word_count\')]\n    if not with_tabular:\n        categories=[c for c in (\'form\',) if c in train]\n    if not numeric:\n        raise ValueError(\'The model has no permitted numeric features.\')\n    return numeric,categories\n\n\ndef matrices(train,predict,numeric,categorical):\n    category_vocab=pd.get_dummies(train[categorical].fillna(\'unknown\').astype(str),dtype=\'float32\')\n    columns=category_vocab.columns.tolist()\n    def convert(frame):\n        values=frame[numeric].replace([np.inf,-np.inf],np.nan).astype(\'float32\')\n        flags=values.isna().astype(\'float32\').add_prefix(\'missing__\')\n        cat=pd.get_dummies(frame[categorical].fillna(\'unknown\').astype(str),dtype=\'float32\')\n        cat=cat.reindex(columns=columns,fill_value=0)\n        return pd.concat([values,flags,cat],axis=1)\n    return convert(train),convert(predict)\n\n\ndef previous_text_links(frame):\n    """Map each filing to the latest strictly earlier same-issuer/form filing."""\n    required={\'adsh\',\'cik\',\'form\',\'filed_date\'}\n    if required-set(frame):\n        raise ValueError(f\'Missing fields for prior-text links: {sorted(required-set(frame))}\')\n    meta=frame[[\'adsh\',\'cik\',\'form\',\'filed_date\']].copy()\n    meta[\'filed_date\']=pd.to_datetime(meta.filed_date,errors=\'raise\')\n    meta[\'cik\']=meta.cik.astype(str).str.lstrip(\'0\').replace(\'\',\'0\')\n    if meta.adsh.duplicated().any():\n        raise ValueError(\'Prior-text mapping requires unique accessions.\')\n    daily=(meta.sort_values([\'cik\',\'form\',\'filed_date\',\'adsh\'],kind=\'stable\')\n           .drop_duplicates([\'cik\',\'form\',\'filed_date\'],keep=\'first\').copy())\n    daily[\'prior_adsh\']=daily.groupby([\'cik\',\'form\'],sort=False).adsh.shift(1)\n    return meta[[\'adsh\',\'cik\',\'form\',\'filed_date\']].merge(\n        daily[[\'cik\',\'form\',\'filed_date\',\'prior_adsh\']],\n        on=[\'cik\',\'form\',\'filed_date\'],how=\'left\',validate=\'many_to_one\')\n\n\ndef text_change_features(frame,batch_size=512):\n    """Compute point-in-time lexical similarity to each issuer\'s prior same-form MD&A."""\n    from sklearn.feature_extraction.text import HashingVectorizer\n    if \'mda_text\' not in frame:\n        raise ValueError(\'MD&A text is required for change features.\')\n    links=previous_text_links(frame)\n    text_by_id=dict(zip(frame.adsh.astype(str),frame.mda_text.fillna(\'\').astype(str)))\n    current=frame.mda_text.fillna(\'\').astype(str).tolist()\n    prior_ids=links.prior_adsh.fillna(\'\').astype(str).tolist()\n    previous=[text_by_id.get(adsh,\'\') for adsh in prior_ids]\n    vectorizer=HashingVectorizer(n_features=2**16,ngram_range=(1,2),norm=\'l2\',\n        alternate_sign=False,lowercase=True,strip_accents=\'unicode\',\n        token_pattern=r\'(?u)\\b[a-zA-Z][a-zA-Z0-9]+\\b\',dtype=np.float32)\n    similarity=np.zeros(len(frame),dtype=np.float32)\n    available=np.zeros(len(frame),dtype=np.int8)\n    for start in range(0,len(frame),batch_size):\n        stop=min(start+batch_size,len(frame))\n        left=current[start:stop]; right=previous[start:stop]\n        mask=np.asarray([bool(a.strip()) and bool(b.strip()) for a,b in zip(left,right)])\n        if mask.any():\n            a=vectorizer.transform([value for value,keep in zip(left,mask) if keep])\n            b=vectorizer.transform([value for value,keep in zip(right,mask) if keep])\n            indices=start+np.flatnonzero(mask)\n            similarity[indices]=np.asarray(a.multiply(b).sum(axis=1)).reshape(-1)\n            available[indices]=1\n    result=links[[\'adsh\',\'prior_adsh\']].copy()\n    result[\'text_prior_available\']=available\n    result[\'text_pairwise_cosine\']=similarity\n    result[\'text_pairwise_novelty\']=np.where(available==1,1.0-similarity,0.0).astype(np.float32)\n    if \'mda_word_count\' in frame:\n        word_counts=pd.Series(frame.mda_word_count.to_numpy(),index=frame.adsh.astype(str))\n    else:\n        counts=np.fromiter((len(value.split()) for value in frame.mda_text.fillna(\'\').astype(str)),\n                           dtype=np.int32,count=len(frame))\n        word_counts=pd.Series(counts,index=frame.adsh.astype(str))\n    prior_counts=links.prior_adsh.map(word_counts).fillna(0).to_numpy(dtype=np.float32)\n    current_counts=np.asarray([len(value.split()) for value in current],dtype=np.float32)\n    result[\'text_word_count_delta\']=current_counts-prior_counts\n    result[\'text_word_count_ratio\']=current_counts/np.maximum(prior_counts,1.0)\n    return result\n\n\ndef add_text_embedding_changes(frame,embeddings):\n    """Add embedding deltas from the strictly earlier same-form filing when available."""\n    links=previous_text_links(frame)\n    region_names=(\'beginning\',\'middle\',\'ending\')\n    region_cols=[[f\'text_emb_{region}_{i}\' for i in range(384)] for region in region_names]\n    required=[\'adsh\']+[column for columns in region_cols for column in columns]\n    if set(required)-set(embeddings):\n        raise ValueError(\'Embedding table lacks expected region vectors.\')\n    current=embeddings[required].copy()\n    current_mean=np.zeros((len(current),384),dtype=np.float32)\n    for columns in region_cols:\n        current_mean+=current[columns].to_numpy(dtype=np.float32)\n    current_mean/=len(region_cols)\n    current_ids=current.adsh.astype(str).to_numpy()\n    prior_table=pd.DataFrame(current_mean,columns=[f\'prior_{i}\' for i in range(384)])\n    prior_table.insert(0,\'prior_adsh\',current_ids)\n    linked=links[[\'adsh\',\'prior_adsh\']].merge(prior_table,on=\'prior_adsh\',how=\'left\',validate=\'many_to_one\')\n    prior=np.column_stack([linked[f\'prior_{i}\'].to_numpy(dtype=np.float32) for i in range(384)])\n    available=(linked.prior_adsh.notna().to_numpy() & np.isfinite(prior).all(axis=1) &\n               (np.linalg.norm(prior,axis=1)>0) & (np.linalg.norm(current_mean,axis=1)>0))\n    prior[~available]=0.0\n    delta=current_mean-prior\n    result=pd.DataFrame(delta,columns=[f\'text_emb_delta_{i}\' for i in range(384)])\n    result.insert(0,\'adsh\',current_ids)\n    result[\'text_embedding_prior_available\']=available.astype(np.int8)\n    return frame[[\'adsh\']].merge(result,on=\'adsh\',how=\'left\',validate=\'one_to_one\')\n\n\ndef fit_xgb(train,predict,numeric,categorical,depth,seed,fixed=None,valid=None,model_params=None):\n    from xgboost import XGBClassifier\n    x,xp=matrices(train,predict,numeric,categorical)\n    y=train.y_amendment_proxy.to_numpy(dtype=np.int8)\n    if len(np.unique(y))!=2:\n        raise ValueError(\'Both labels are required for fitting.\')\n    params={\'max_depth\':depth,\'learning_rate\':.025,\'min_child_weight\':5,\'subsample\':.85,\n            \'colsample_bytree\':.85,\'reg_lambda\':8,\'reg_alpha\':.2}\n    params.update(model_params or {})\n    model=XGBClassifier(n_estimators=fixed or 1600,**params,\n        tree_method=\'hist\',device=\'cuda\',n_jobs=4,objective=\'binary:logistic\',eval_metric=\'aucpr\',\n        random_state=seed,**({\'early_stopping_rounds\':120} if fixed is None else {}))\n    if fixed is None:\n        if valid is None:\n            raise ValueError(\'Chronological validation is required for early stopping.\')\n        _,v=matrices(train,valid,numeric,categorical)\n        model.fit(x,y,eval_set=[(v,valid.y_amendment_proxy.to_numpy(dtype=np.int8))],verbose=False)\n        iterations=int(model.best_iteration)+1\n    else:\n        model.fit(x,y,verbose=False); iterations=int(fixed)\n    device=json.loads(model.get_booster().save_config())[\'learner\'][\'generic_param\'][\'device\']\n    if not device.startswith(\'cuda\'):\n        raise RuntimeError(f\'XGBoost used {device}; this notebook requires GPU fitting.\')\n    probabilities=model.predict_proba(xp)[:,1]\n    return probabilities,iterations,model,x.columns.tolist()\n\n\ndef _rank_groups(frame):\n    part=frame.copy()\n    part[\'_rank_year\']=pd.to_datetime(part.filed_date).dt.year\n    order=np.lexsort((part.adsh.astype(str).to_numpy(),part.form.astype(str).to_numpy(),\n                      part._rank_year.to_numpy()))\n    ordered=part.iloc[order]\n    sizes=ordered.groupby([\'_rank_year\',\'form\'],sort=False).size().astype(int).tolist()\n    labels=ordered.y_amendment_proxy.to_numpy(dtype=np.int8)\n    start=0\n    for size in sizes:\n        if size<2 or len(np.unique(labels[start:start+size]))<2:\n            raise ValueError(\'Pairwise ranking needs both labels in every year-form query group.\')\n        start+=size\n    return order,sizes\n\n\ndef fit_pairwise_ranker(train,predict,numeric,categorical,depth,seed,fixed=None,valid=None):\n    """GPU XGBoost pairwise ranking within filing-year/form matched queries."""\n    from xgboost import XGBRanker\n    x,xp=matrices(train,predict,numeric,categorical)\n    train_order,train_groups=_rank_groups(train)\n    y=train.iloc[train_order].y_amendment_proxy.to_numpy(dtype=np.int8)\n    params={\'max_depth\':depth,\'learning_rate\':.03,\'min_child_weight\':6,\'subsample\':.8,\n            \'colsample_bytree\':.8,\'reg_lambda\':12,\'reg_alpha\':.3}\n    ranker=XGBRanker(n_estimators=fixed or 1600,**params,tree_method=\'hist\',device=\'cuda\',\n        n_jobs=4,objective=\'rank:pairwise\',eval_metric=\'ndcg\',random_state=seed,\n        **({\'early_stopping_rounds\':120} if fixed is None else {}))\n    if fixed is None:\n        if valid is None:\n            raise ValueError(\'Pairwise ranker early stopping requires a matched validation cohort.\')\n        _,v=matrices(train,valid,numeric,categorical)\n        valid_order,valid_groups=_rank_groups(valid)\n        vy=valid.iloc[valid_order].y_amendment_proxy.to_numpy(dtype=np.int8)\n        ranker.fit(x.iloc[train_order],y,group=train_groups,\n                   eval_set=[(v.iloc[valid_order],vy)],eval_group=[valid_groups],verbose=False)\n        iterations=int(ranker.best_iteration)+1\n    else:\n        ranker.fit(x.iloc[train_order],y,group=train_groups,verbose=False)\n        iterations=int(fixed)\n    device=json.loads(ranker.get_booster().save_config())[\'learner\'][\'generic_param\'][\'device\']\n    if not device.startswith(\'cuda\'):\n        raise RuntimeError(f\'XGBoost ranker used {device}; GPU fitting is required.\')\n    margin=ranker.predict(xp)\n    probabilities=1.0/(1.0+np.exp(-np.clip(margin,-30,30)))\n    return probabilities,iterations,ranker,x.columns.tolist()\n\n\ndef fit_tfidf_xgb(train,predict,numeric,categorical,depth,seed,fixed=None,valid=None,\n                  save_path=None,model_params=None):\n    """Fit sparse word/character TF-IDF plus tabular features on CUDA XGBoost."""\n    from scipy import sparse\n    from sklearn.feature_extraction.text import TfidfVectorizer\n    from sklearn.pipeline import FeatureUnion\n    from xgboost import XGBClassifier\n    vectorizer=FeatureUnion([\n        (\'word\',TfidfVectorizer(analyzer=\'word\',ngram_range=(1,2),min_df=3,\n            max_features=40000,sublinear_tf=True,dtype=np.float32,strip_accents=\'unicode\')),\n        (\'char\',TfidfVectorizer(analyzer=\'char_wb\',ngram_range=(3,5),min_df=4,\n            max_features=30000,sublinear_tf=True,dtype=np.float32,strip_accents=\'unicode\'))])\n    train_text=train.mda_text.fillna(\'\').astype(str).tolist()\n    predict_text=predict.mda_text.fillna(\'\').astype(str).tolist()\n    text_x=vectorizer.fit_transform(train_text)\n    text_p=vectorizer.transform(predict_text)\n    dense_x,dense_p=matrices(train,predict,numeric,categorical)\n    x=sparse.hstack([text_x,sparse.csr_matrix(dense_x.to_numpy(dtype=np.float32))],format=\'csr\')\n    xp=sparse.hstack([text_p,sparse.csr_matrix(dense_p.to_numpy(dtype=np.float32))],format=\'csr\')\n    params={\'max_depth\':depth,\'learning_rate\':.04,\'min_child_weight\':8,\'subsample\':.8,\n            \'colsample_bytree\':.6,\'reg_lambda\':16,\'reg_alpha\':.5}\n    params.update(model_params or {})\n    model=XGBClassifier(n_estimators=fixed or 1400,**params,tree_method=\'hist\',device=\'cuda\',\n        n_jobs=4,objective=\'binary:logistic\',eval_metric=\'aucpr\',random_state=seed,\n        **({\'early_stopping_rounds\':120} if fixed is None else {}))\n    if fixed is None:\n        if valid is None:\n            raise ValueError(\'Sparse text XGBoost early stopping requires a time validation cohort.\')\n        text_v=vectorizer.transform(valid.mda_text.fillna(\'\').astype(str).tolist())\n        _,dense_v=matrices(train,valid,numeric,categorical)\n        v=sparse.hstack([text_v,sparse.csr_matrix(dense_v.to_numpy(dtype=np.float32))],format=\'csr\')\n        model.fit(x,train.y_amendment_proxy.to_numpy(dtype=np.int8),\n                  eval_set=[(v,valid.y_amendment_proxy.to_numpy(dtype=np.int8))],verbose=False)\n        iterations=int(model.best_iteration)+1\n    else:\n        model.fit(x,train.y_amendment_proxy.to_numpy(dtype=np.int8),verbose=False)\n        iterations=int(fixed)\n    device=json.loads(model.get_booster().save_config())[\'learner\'][\'generic_param\'][\'device\']\n    if not device.startswith(\'cuda\'):\n        raise RuntimeError(f\'Sparse XGBoost used {device}; GPU fitting is required.\')\n    if save_path is not None:\n        import joblib\n        joblib.dump(vectorizer,Path(str(save_path)+\'.tfidf.joblib\'))\n    return model.predict_proba(xp)[:,1],iterations,model,[]\n\n\ndef fit_mlp(train_x,y,predict_x,seed,valid_x=None,valid_y=None,epochs=None,save_path=None):\n    """Fit a small dense classifier with early stopping on an inner time slice."""\n    import torch\n    from sklearn.metrics import average_precision_score\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'The MLP requires CUDA; CPU fallback is disabled.\')\n    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)\n    a=train_x.to_numpy(dtype=np.float32); b=predict_x.to_numpy(dtype=np.float32)\n    usable=~np.isnan(a).all(axis=0)\n    median=np.zeros(a.shape[1],dtype=np.float32)\n    median[usable]=np.nanmedian(a[:,usable],axis=0)\n    a=np.where(np.isnan(a),median,a); b=np.where(np.isnan(b),median,b)\n    mean=a.mean(axis=0); std=np.maximum(a.std(axis=0),1e-3)\n    a=np.clip((a-mean)/std,-10,10); b=np.clip((b-mean)/std,-10,10)\n    if valid_x is not None:\n        v=valid_x.to_numpy(dtype=np.float32)\n        v=np.where(np.isnan(v),median,v); v=np.clip((v-mean)/std,-10,10)\n    device=torch.device(\'cuda\')\n    model=torch.nn.Sequential(\n        torch.nn.Linear(a.shape[1],256),torch.nn.LayerNorm(256),torch.nn.GELU(),torch.nn.Dropout(.35),\n        torch.nn.Linear(256,128),torch.nn.LayerNorm(128),torch.nn.GELU(),torch.nn.Dropout(.2),\n        torch.nn.Linear(128,1)).to(device)\n    xa=torch.tensor(a,device=device); ya=torch.tensor(y,dtype=torch.float32,device=device)\n    xb=torch.tensor(b,device=device)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=8e-4,weight_decay=.02)\n    best=-np.inf; best_epoch=0; best_state=None; stale=0\n    def probabilities(matrix):\n        model.eval()\n        with torch.no_grad():\n            return np.concatenate([torch.sigmoid(model(matrix[i:i+2048])).flatten().cpu().numpy()\n                                   for i in range(0,len(matrix),2048)])\n    for epoch in range(epochs or 100):\n        model.train(); order=torch.randperm(len(xa),device=device)\n        for start in range(0,len(order),256):\n            indices=order[start:start+256]\n            loss=torch.nn.functional.binary_cross_entropy_with_logits(\n                model(xa[indices]).flatten(),ya[indices])\n            optimizer.zero_grad(set_to_none=True); loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(),5); optimizer.step()\n        if epochs is None:\n            if valid_x is None or valid_y is None or len(np.unique(valid_y))<2:\n                raise ValueError(\'MLP early stopping requires both classes in an inner validation slice.\')\n            score=average_precision_score(valid_y,probabilities(torch.tensor(v,device=device)))\n            if score>best+1e-5:\n                best=score; best_epoch=epoch+1; stale=0\n                best_state={name:value.detach().clone() for name,value in model.state_dict().items()}\n            else:\n                stale+=1\n            if stale>=12: break\n    if epochs is None:\n        if best_state is None:\n            raise RuntimeError(\'MLP did not retain an early-stopping checkpoint.\')\n        model.load_state_dict(best_state)\n        chosen=best_epoch\n    else:\n        chosen=int(epochs)\n    probabilities_out=probabilities(xb)\n    if save_path is not None:\n        save_path=Path(save_path); save_path.parent.mkdir(parents=True,exist_ok=True)\n        torch.save({\'state_dict\':{name:value.detach().cpu() for name,value in model.state_dict().items()},\n                    \'feature_columns\':train_x.columns.tolist(),\'median\':median,\'mean\':mean,\'std\':std,\n                    \'seed\':int(seed),\'epochs\':int(chosen),\'hidden_layers\':[256,128]},save_path)\n    del model,xa,ya,xb\n    torch.cuda.empty_cache()\n    return probabilities_out,chosen\n\n\ndef specialist_model_base(base,form):\n    return Path(base).with_name(Path(base).name+\'_\'+str(form).replace(\'-\',\'\'))\n\n\ndef fit_model(config,train,predict,numeric,categorical,seed,fixed=None,valid=None,save_path=None):\n    if config[\'family\']==\'form_specialists\':\n        probabilities=np.full(len(predict),np.nan,dtype=np.float32)\n        iterations=[]; columns=[]\n        for form in sorted(set(train.form.astype(str)) & set(predict.form.astype(str))):\n            train_mask=train.form.astype(str).eq(form)\n            predict_mask=predict.form.astype(str).eq(form).to_numpy()\n            valid_part=valid.loc[valid.form.astype(str).eq(form)].copy() if valid is not None else None\n            base=specialist_model_base(save_path,form) if save_path is not None else None\n            specialized={**config,\'family\':\'xgb\'}\n            pred,count,feature_columns=fit_model(\n                specialized,train.loc[train_mask].copy(),predict.loc[predict_mask].copy(),\n                numeric,categorical,seed,fixed=fixed,valid=valid_part,save_path=base)\n            probabilities[predict_mask]=pred\n            iterations.append(count); columns.extend(f\'{form}:{name}\' for name in feature_columns)\n        if not np.isfinite(probabilities).all():\n            raise ValueError(\'Form specialists did not score every requested filing.\')\n        return probabilities,int(np.median(iterations)),columns\n    if config[\'family\']==\'xgb\':\n        probabilities,iterations,model,columns=fit_xgb(\n            train,predict,numeric,categorical,config[\'depth\'],seed,fixed=fixed,valid=valid,\n            model_params=config.get(\'xgb_params\'))\n        if save_path is not None:\n            model.save_model(str(save_path.with_suffix(\'.json\')))\n        return probabilities,iterations,columns\n    if config[\'family\']==\'ranker\':\n        probabilities,iterations,model,columns=fit_pairwise_ranker(\n            train,predict,numeric,categorical,config[\'depth\'],seed,fixed=fixed,valid=valid)\n        if save_path is not None:\n            model.save_model(str(save_path.with_suffix(\'.json\')))\n        return probabilities,iterations,columns\n    if config[\'family\']==\'tfidf_xgb\':\n        probabilities,iterations,model,columns=fit_tfidf_xgb(\n            train,predict,numeric,categorical,config[\'depth\'],seed,fixed=fixed,valid=valid,\n            save_path=save_path,model_params=config.get(\'xgb_params\'))\n        if save_path is not None:\n            model.save_model(str(save_path.with_suffix(\'.json\')))\n        return probabilities,iterations,columns\n    x,xp=matrices(train,predict,numeric,categorical)\n    vx=None; vy=None\n    if fixed is None:\n        if valid is None:\n            raise ValueError(\'Chronological inner validation is required for MLP early stopping.\')\n        _,vx=matrices(train,valid,numeric,categorical)\n        vy=valid.y_amendment_proxy.to_numpy(dtype=np.int8)\n    probabilities,iterations=fit_mlp(x,train.y_amendment_proxy.to_numpy(dtype=np.int8),xp,\n                                      seed,valid_x=vx,valid_y=vy,epochs=fixed,\n                                      save_path=save_path.with_suffix(\'.pt\') if save_path else None)\n    if save_path is not None:\n        save_path.with_suffix(\'.json\').write_text(json.dumps(\n            {\'feature_columns\':x.columns.tolist(),\'seed\':int(seed),\'epochs\':int(iterations),\n             \'device\':\'cuda\',\'family\':\'mlp\'},indent=2),encoding=\'utf-8\')\n    return probabilities,iterations,x.columns.tolist()\n\n\ndef score_report(y,p,threshold):\n    from sklearn.metrics import average_precision_score,roc_auc_score,matthews_corrcoef\n    y=np.asarray(y,dtype=np.int8); p=np.asarray(p,dtype=float)\n    threshold_array=np.asarray(threshold,dtype=float)\n    if threshold_array.ndim and threshold_array.shape!=p.shape:\n        raise ValueError(\'Per-row thresholds must align with prediction scores.\')\n    pred=p>=threshold_array\n    threshold_report=(float(threshold_array) if threshold_array.ndim==0 else \'row_specific\')\n    tp=int(np.sum(pred&(y==1))); fp=int(np.sum(pred&(y==0)))\n    fn=int(np.sum(~pred&(y==1))); tn=int(np.sum(~pred&(y==0)))\n    precision=tp/max(tp+fp,1); recall=tp/max(tp+fn,1)\n    f1=2*precision*recall/max(precision+recall,1e-12)\n    accuracy=(tp+tn)/len(y)\n    return {\'n\':len(y),\'positive_count\':int(y.sum()),\'prevalence_in_matched_sample\':float(y.mean()),\n            \'threshold\':threshold_report,\'accuracy\':accuracy,\'precision\':precision,\'recall\':recall,\n            \'f1\':f1,\'balanced_accuracy\':(recall+tn/max(tn+fp,1))/2,\n            \'pr_auc\':float(average_precision_score(y,p)),\'roc_auc\':float(roc_auc_score(y,p)),\n            \'mcc\':float(matthews_corrcoef(y,pred)),\'tn_fp_fn_tp\':[tn,fp,fn,tp],\n            \'all_four_ge_80\':bool(min(accuracy,precision,recall,f1)>=.80),\n            \'all_four_ge_85\':bool(min(accuracy,precision,recall,f1)>=.85)}\n\n\ndef issuer_cluster_confidence_intervals(frame,probability,threshold,draws=400,seed=2026):\n    """Issuer-cluster bootstrap intervals for thresholded matched-cohort metrics."""\n    probability=np.asarray(probability,dtype=float)\n    if len(frame)!=len(probability) or frame.cik.isna().any():\n        raise ValueError(\'Cluster intervals require one finite score and CIK per row.\')\n    y=frame.y_amendment_proxy.to_numpy(dtype=np.int8)\n    threshold_array=np.asarray(threshold,dtype=float)\n    if threshold_array.ndim and threshold_array.shape!=probability.shape:\n        raise ValueError(\'Per-row bootstrap thresholds must align with prediction scores.\')\n    pred=probability>=threshold_array\n    counts=pd.DataFrame({\'cik\':frame.cik.astype(str).to_numpy(),\n        \'tp\':pred&(y==1),\'fp\':pred&(y==0),\'tn\':(~pred)&(y==0),\'fn\':(~pred)&(y==1)})\n    clusters=counts.groupby(\'cik\',sort=False)[[\'tp\',\'fp\',\'tn\',\'fn\']].sum().to_numpy(dtype=np.int64)\n    if len(clusters)<2 or draws<1:\n        raise ValueError(\'Cluster intervals need at least two issuers and one bootstrap draw.\')\n    rng=np.random.default_rng(seed)\n    estimates=[]\n    for _ in range(draws):\n        tp,fp,tn,fn=clusters[rng.integers(0,len(clusters),len(clusters))].sum(axis=0)\n        precision=tp/max(tp+fp,1); recall=tp/max(tp+fn,1)\n        f1=2*precision*recall/max(precision+recall,1e-12)\n        estimates.append([(tp+tn)/max(tp+fp+tn+fn,1),precision,recall,f1])\n    bounds=np.quantile(np.asarray(estimates),[.025,.975],axis=0)\n    return {\'method\':\'issuer-cluster bootstrap percentile interval\',\'draws\':int(draws),\n            \'clusters\':int(len(clusters)),\n            \'ci_95\':{name:[float(bounds[0,i]),float(bounds[1,i])]\n                     for i,name in enumerate((\'accuracy\',\'precision\',\'recall\',\'f1\'))}}\n\n\ndef calibrated_form_thresholds(calibration,probability,global_threshold,min_per_class=30):\n    """Calibrate 10-K/10-Q cutoffs on 2023 only, with a minimum-count fallback."""\n    probability=np.asarray(probability,dtype=float)\n    if len(calibration)!=len(probability):\n        raise ValueError(\'Calibration rows and scores differ in length.\')\n    result={}\n    forms=calibration.form.astype(str).to_numpy()\n    labels=calibration.y_amendment_proxy.to_numpy(dtype=np.int8)\n    for form in sorted(np.unique(forms)):\n        mask=forms==form\n        counts=np.bincount(labels[mask],minlength=2)\n        if np.min(counts)<min_per_class:\n            result[form]={\'threshold\':float(global_threshold),\'source\':\'global_fallback\',\n                          \'calibration_rows\':int(mask.sum()),\n                          \'positive_count\':int(counts[1]),\'negative_count\':int(counts[0])}\n        else:\n            selected=core.threshold_choice(labels[mask],probability[mask])\n            result[form]={**selected,\'source\':\'form_specific_2023_calibration\',\n                          \'calibration_rows\':int(mask.sum()),\n                          \'positive_count\':int(counts[1]),\'negative_count\':int(counts[0])}\n    return result\n\n\ndef thresholds_for_frame(frame,threshold_map,global_threshold):\n    return np.asarray([threshold_map.get(str(form),{}).get(\'threshold\',global_threshold)\n                       for form in frame.form],dtype=float)\n\n\ndef embed_text(frame,cache_dir,device):\n    import torch\n    from sentence_transformers import SentenceTransformer\n    from huggingface_hub import model_info\n    cache_dir=Path(cache_dir); cache_dir.mkdir(parents=True,exist_ok=True)\n    revision=model_info(\'sentence-transformers/all-MiniLM-L6-v2\').sha\n    encoder_record={\'model\':\'sentence-transformers/all-MiniLM-L6-v2\',\'revision\':revision,\n                    \'license\':\'Apache-2.0\',\'source\':\'https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2\',\n                    \'pooling\':\'Up to 20 chunks sampled evenly across the full extracted section; beginning, middle, ending means and document-level max pooling\'}\n    (cache_dir/\'encoder.json\').write_text(json.dumps(encoder_record,indent=2),encoding=\'utf-8\')\n    cache_key=hashlib.sha256(json.dumps([core.file_hash(frame.attrs[\'dataset_path\']),\n                                        revision,\'mda-full-section-streamed-even-chunks-regions-max-v5\'],sort_keys=True).encode()).hexdigest()[:24]\n    path=cache_dir/f\'{cache_key}.parquet\'\n    if path.exists():\n        cached=pd.read_parquet(path)\n        if cached.adsh.tolist()==frame.adsh.tolist():\n            return cached,revision\n    encoder=SentenceTransformer(\'sentence-transformers/all-MiniLM-L6-v2\',revision=revision,\n                                device=device,trust_remote_code=False)\n    if not str(encoder.device).startswith(\'cuda\'):\n        raise RuntimeError(f\'MiniLM is on {encoder.device}; this experiment requires GPU embeddings.\')\n    # Stream at most a few thousand 160-word chunks at once. Materializing\n    # token lists/chunks for every filing can exceed Colab RAM on the full\n    # manifest even when the GPU itself has enough memory.\n    vectors=np.zeros((len(frame),3,384),dtype=np.float32)\n    counts=np.zeros((len(frame),3),dtype=np.float32)\n    chunk_max=np.full((len(frame),384),-np.inf,dtype=np.float32)\n    chunks=[]; owners=[]; regions=[]; processed=0\n    def flush_chunk_batch():\n        nonlocal processed\n        if not chunks:\n            return\n        encoded=encoder.encode(chunks,batch_size=64,show_progress_bar=False,\n                              convert_to_numpy=True,normalize_embeddings=True)\n        owner_ids=np.asarray(owners,dtype=np.int64); region_ids=np.asarray(regions,dtype=np.int64)\n        np.add.at(vectors,(owner_ids,region_ids),encoded)\n        np.add.at(counts,(owner_ids,region_ids),1)\n        np.maximum.at(chunk_max,owner_ids,encoded)\n        processed+=len(chunks)\n        print(f\'Embedded {processed:,} MD&A text chunks on GPU.\',flush=True)\n        chunks.clear(); owners.clear(); regions.clear()\n    for owner,text in enumerate(frame.mda_text.fillna(\'\').astype(str)):\n        tokens=text.split()\n        chunk_count=(len(tokens)+159)//160\n        if not chunk_count:\n            continue\n        starts=list(range(0,len(tokens),160))\n        selected=(np.unique(np.linspace(0,len(starts)-1,num=20,dtype=int)).tolist()\n                  if len(starts)>20 else list(range(len(starts))))\n        edge_count=max(1,(chunk_count+3)//4)\n        for chunk_index in selected:\n            start=starts[chunk_index]\n            part=\' \'.join(tokens[start:start+160])\n            if not part:\n                continue\n            chunks.append(part); owners.append(owner)\n            regions.append(0 if chunk_index<edge_count else\n                           2 if chunk_index>=chunk_count-edge_count else 1)\n            if len(chunks)>=4096:\n                flush_chunk_batch()\n    flush_chunk_batch()\n    vectors/=np.maximum(counts[:,:,None],1)\n    chunk_max[~np.isfinite(chunk_max)]=0.0\n    pieces=[frame[[\'adsh\']].reset_index(drop=True)]\n    for region,name in enumerate((\'beginning\',\'middle\',\'ending\')):\n        pieces.append(pd.DataFrame(vectors[:,region,:],\n                                   columns=[f\'text_emb_{name}_{i}\' for i in range(384)]))\n    pieces.append(pd.DataFrame(chunk_max,columns=[f\'text_emb_chunk_max_{i}\' for i in range(384)]))\n    result=pd.concat(pieces,axis=1)\n    temp=path.with_suffix(\'.part.parquet\'); result.to_parquet(temp,index=False); temp.replace(path)\n    del encoder; torch.cuda.empty_cache()\n    return result,revision\n\n\ndef text_style_features(frame):\n    """Transparent style counts with bounded per-document token memory."""\n    texts=frame.mda_text.fillna(\'\').astype(str).tolist()\n    pattern=re.compile(r"[A-Za-z]+(?:\'[A-Za-z]+)?|\\d+(?:[.,]\\d+)?%?")\n    groups={\n        \'uncertainty\':{\'may\',\'might\',\'could\',\'approximately\',\'estimate\',\'estimated\',\'uncertain\',\'uncertainty\'},\n        \'forward_looking\':{\'expect\',\'expects\',\'expected\',\'anticipate\',\'anticipates\',\'believe\',\'plan\',\'plans\',\'intend\',\'intends\'},\n        \'negation\':{\'no\',\'not\',\'never\',\'neither\',\'nor\',\'without\',\'unable\'},\n        \'contrast\':{\'however\',\'although\',\'whereas\',\'despite\',\'nevertheless\',\'conversely\'} }\n    known_counts=(frame.mda_word_count.to_numpy(dtype=np.float32)\n                  if \'mda_word_count\' in frame else None)\n    values=[]\n    for i,text in enumerate(texts):\n        words=text.split()\n        if len(words)>2000:\n            middle=len(words)//2\n            selected=words[:700]+words[middle-350:middle+350]+words[-600:]\n            full_count=known_counts[i] if known_counts is not None else len(words)\n        else:\n            selected=words\n            full_count=known_counts[i] if known_counts is not None else len(words)\n        tokens=pattern.findall(\' \'.join(selected))\n        lowered=[token.lower() for token in tokens]\n        denom=max(len(tokens),1)\n        row={\n            \'text_style_unique_token_fraction\':len(set(lowered))/denom,\n            \'text_style_mean_token_length\':float(np.mean([len(token) for token in tokens])) if tokens else 0.0,\n            \'text_style_long_token_fraction\':sum(len(token)>=10 for token in tokens)/denom,\n            \'text_style_numeric_token_fraction\':sum(any(ch.isdigit() for ch in token) for token in tokens)/denom,\n            \'text_style_mean_sentence_length\':float(full_count)/max(text.count(\'.\')+text.count(\'!\')+text.count(\'?\'),1),\n        }\n        for name,vocabulary in groups.items():\n            row[f\'text_style_{name}_per_1000\']=1000*sum(token in vocabulary for token in lowered)/denom\n        values.append(row)\n    result=pd.DataFrame(values,index=frame.index)\n    # Transparent generic language groups; no external/private lexicon is used.\n    return result.astype(\'float32\')\n\n\ndef prior_filing_features(frame):\n    """Build same-issuer, same-form changes from the complete prepared panel."""\n    columns=[name for name in core.RATIOS if name in frame]\n    if not columns:\n        return frame[[\'adsh\']].copy()\n    keys=[\'cik\',\'form\',\'filed_date\']\n    daily=(frame.groupby(keys,as_index=False,dropna=False)[columns].mean()\n           .sort_values(keys,kind=\'stable\').reset_index(drop=True))\n    previous=daily.groupby([\'cik\',\'form\'],sort=False)[columns].shift(1)\n    engineered=daily[keys].copy()\n    for name in columns:\n        prior=previous[name].to_numpy(dtype=\'float64\')\n        current=daily[name].to_numpy(dtype=\'float64\')\n        delta=current-prior\n        engineered[f\'prior_filing_{name}\']=prior\n        engineered[f\'change_prior_filing_{name}\']=delta\n        engineered[f\'rel_change_prior_filing_{name}\']=np.clip(\n            delta/(np.abs(prior)+1e-3),-10,10)\n    mapping=frame[[\'adsh\']+keys].merge(engineered,on=keys,how=\'left\',validate=\'many_to_one\')\n    return mapping.drop(columns=keys)\n\n\ndef run(root,text_path=None,experiment_name=\'mda_text_expanded\'):\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Select a Colab NVIDIA GPU runtime before running this experiment.\')\n    root=Path(root)\n    text_path=Path(text_path) if text_path else root/\'data/colab_mda_text_case_control.csv.gz\'\n    proxy_path=root/\'data/colab_amendment_proxy.csv.gz\'\n    if not text_path.is_file() or not proxy_path.is_file():\n        raise FileNotFoundError(\'Upload both prepared CSV files to Drive under data/.\')\n    gpu_name=torch.cuda.get_device_name(0)\n    print(\'Confirmed GPU:\',gpu_name,flush=True)\n    runtime_versions={\'python\':sys.version.split()[0],**{\n        name:importlib.metadata.version(dist) for name,dist in\n        [(\'torch\',\'torch\'),(\'xgboost\',\'xgboost-cu12\'),(\'scikit_learn\',\'scikit-learn\'),\n         (\'sentence_transformers\',\'sentence-transformers\'),(\'numpy\',\'numpy\'),(\'pandas\',\'pandas\')]}}\n    runtime_signature={**runtime_versions,\'gpu\':gpu_name}\n    text=pd.read_csv(text_path,compression=\'gzip\',dtype={\'adsh\':str,\'cik\':str,\'form\':str},low_memory=False)\n    text[\'filed_date\']=pd.to_datetime(text.filed_date,errors=\'raise\')\n    text[\'mda_text\']=text.mda_text.fillna(\'\').astype(str)\n    if \'mda_word_count\' not in text:\n        text[\'mda_word_count\']=text.mda_text.str.split().str.len()\n    if \'text_available\' not in text:\n        text[\'text_available\']=text.mda_word_count.ge(200).astype(\'int8\')\n    text[\'y_amendment_proxy\']=text.y_proxy.astype(\'int8\')\n    # Reuse current as-filed features only for the selected MD&A accessions.\n    wanted=set(text.adsh); parts=[]; history_parts=[]\n    proxy_row_count=0; proxy_positive_count=0\n    history_columns=[name for name in core.RATIOS if name in pd.read_csv(\n        proxy_path,compression=\'gzip\',nrows=0).columns]\n    for chunk in pd.read_csv(proxy_path,compression=\'gzip\',dtype={\'adsh\':str,\'cik\':str},\n                             low_memory=False,chunksize=50000):\n        proxy_row_count+=len(chunk)\n        proxy_positive_count+=int(chunk.y_amendment_proxy.sum())\n        keep=chunk[chunk.adsh.isin(wanted)]\n        if len(keep): parts.append(keep)\n        history_parts.append(chunk[[\'adsh\',\'cik\',\'form\',\'filed_date\',*history_columns]])\n    tabular=pd.concat(parts,ignore_index=True)\n    if tabular.adsh.duplicated().any() or set(tabular.adsh)!=wanted:\n        raise ValueError(\'MD&A accessions do not match the current prepared data.\')\n    aligned=text[[\'adsh\',\'form\',\'filed_date\',\'y_amendment_proxy\']].merge(\n        tabular[[\'adsh\',\'form\',\'filed_date\',\'y_amendment_proxy\']],on=\'adsh\',\n        how=\'inner\',validate=\'one_to_one\',suffixes=(\'_text\',\'_proxy\'))\n    if not aligned.form_text.eq(aligned.form_proxy).all() or not aligned.y_amendment_proxy_text.eq(aligned.y_amendment_proxy_proxy).all():\n        raise ValueError(\'Filing metadata or relabeling mismatch.\')\n    if not aligned.filed_date_text.eq(aligned.filed_date_proxy).all():\n        raise ValueError(\'Filing dates differ between the text sample and proxy data.\')\n    dataset_hash=core.file_hash(proxy_path); text_hash=core.file_hash(text_path)\n    text.attrs[\'dataset_path\']=text_path\n    embeddings,revision=embed_text(text,root/\'evidence_v5/mda_embeddings\',device=\'cuda\')\n    style=text_style_features(text)\n    lexical_change=text_change_features(text)\n    sample=text.merge(embeddings,on=\'adsh\',validate=\'one_to_one\').merge(\n        tabular.drop(columns=[\'cik\',\'form\',\'filed_date\',\'y_amendment_proxy\']),on=\'adsh\',\n        how=\'inner\',validate=\'one_to_one\',suffixes=(\'\',\'_proxy\'))\n    sample=pd.concat([sample.reset_index(drop=True),style.reset_index(drop=True),\n                      lexical_change.drop(columns=\'adsh\').reset_index(drop=True)],axis=1)\n    embedding_change=add_text_embedding_changes(sample,embeddings)\n    sample=sample.merge(embedding_change,on=\'adsh\',how=\'left\',validate=\'one_to_one\')\n    history=pd.concat(history_parts,ignore_index=True)\n    history[\'filed_date\']=pd.to_datetime(history.filed_date,errors=\'raise\')\n    sample=sample.merge(prior_filing_features(history),on=\'adsh\',how=\'left\',validate=\'one_to_one\')\n    # Use labels and dates checked against the current CSV above.\n    sample[\'y_amendment_proxy\']=sample.y_proxy.astype(\'int8\')\n    sample[\'filed_date\']=pd.to_datetime(sample.filed_date,errors=\'raise\')\n    sample[\'fp\']=sample.fp.fillna(\'unknown\').astype(str)\n    sample[\'sic2\']=sample.sic2.fillna(\'unknown\').astype(str)\n    for name in (\'y_amendment_proxy\',\'cik\'):\n        if name not in sample: raise ValueError(f\'Missing required field {name}\')\n    # A fixed CIK hash keeps issuers out of every CV fit and out of final model\n    # selection; their later filings provide a separate cold-start comparison.\n    heldout_issuer=issuer_holdout_mask(sample.cik)\n    selection_sample=sample.loc[~heldout_issuer].copy()\n    configs=[{\'name\':\'text_xgb_d3\',\'family\':\'xgb\',\'depth\':3,\'tabular\':False},\n             {\'name\':\'text_xgb_d5\',\'family\':\'xgb\',\'depth\':5,\'tabular\':False},\n             {\'name\':\'text_plus_financial_xgb_d3\',\'family\':\'xgb\',\'depth\':3,\'tabular\':True},\n             {\'name\':\'text_plus_financial_xgb_d5\',\'family\':\'xgb\',\'depth\':5,\'tabular\':True},\n             {\'name\':\'text_plus_financial_xgb_d2_regularized\',\'family\':\'xgb\',\'depth\':2,\'tabular\':True,\n              \'xgb_params\':{\'min_child_weight\':8,\'subsample\':.8,\'colsample_bytree\':.7,\n                            \'reg_lambda\':20,\'reg_alpha\':.5}},\n             {\'name\':\'text_plus_financial_xgb_d4\',\'family\':\'xgb\',\'depth\':4,\'tabular\':True,\n              \'xgb_params\':{\'learning_rate\':.02,\'min_child_weight\':6,\'colsample_bytree\':.8,\n                            \'reg_lambda\':12,\'reg_alpha\':.3}},\n             {\'name\':\'text_plus_financial_xgb_d6_regularized\',\'family\':\'xgb\',\'depth\':6,\'tabular\':True,\n              \'xgb_params\':{\'min_child_weight\':10,\'subsample\':.8,\'colsample_bytree\':.7,\n                            \'reg_lambda\':16,\'reg_alpha\':.5}},\n             {\'name\':\'text_plus_financial_xgb_d2_form_specialists\',\'family\':\'form_specialists\',\'depth\':2,\'tabular\':True,\n              \'xgb_params\':{\'min_child_weight\':8,\'subsample\':.8,\'colsample_bytree\':.7,\n                            \'reg_lambda\':20,\'reg_alpha\':.5}},\n             {\'name\':\'text_plus_financial_pairwise_ranker\',\'family\':\'ranker\',\'depth\':3,\'tabular\':True},\n             {\'name\':\'text_plus_financial_word_char_tfidf_xgb\',\'family\':\'tfidf_xgb\',\'depth\':3,\'tabular\':True,\n              \'xgb_params\':{\'min_child_weight\':10,\'subsample\':.8,\'colsample_bytree\':.6,\n                            \'reg_lambda\':16,\'reg_alpha\':.5}},\n             {\'name\':\'text_mlp\',\'family\':\'mlp\',\'depth\':0,\'tabular\':False},\n             {\'name\':\'text_plus_financial_mlp\',\'family\':\'mlp\',\'depth\':0,\'tabular\':True}]\n    # The early-stopping year is inside each training window; the later year\n    # remains untouched until outer scoring.\n    folds=[(2016,2018),(2018,2020),(2020,2022)]\n    cv={}; leaderboard=[]\n    from sklearn.metrics import average_precision_score\n    trainer_hash=core.file_hash(__file__)\n    core_hash=core.file_hash(core.__file__)\n    cv_cache_root=root/\'evidence_v5\'/\'mda_cv_cache\'\n    for config in configs:\n        numeric,categorical=columns_for(sample,config[\'tabular\'])\n        preds=[]; iterations=[]; fold_scores=[]\n        for fit_year,valid_year in folds:\n            first_year=max(2009,fit_year-7)\n            fit_rows=selection_sample[selection_sample.filed_date.between(f\'{first_year}-01-01\',f\'{fit_year-1}-12-31\')].copy()\n            stop_pool=selection_sample[selection_sample.filed_date.between(f\'{fit_year}-01-01\',f\'{fit_year}-12-31\')]\n            if fit_rows.empty or stop_pool.empty:\n                raise ValueError(f\'Empty inner time split for {fit_year} -> {valid_year}.\')\n            validation=matched_cohort(selection_sample[selection_sample.filed_date.dt.year.eq(valid_year)],707)\n            train=balanced_train(fit_rows,42)\n            stop=matched_cohort(stop_pool,707)\n            fold_dir=cv_cache_root/config[\'name\']/f\'{fit_year}_{valid_year}\'\n            probability_path=fold_dir/\'probabilities.npy\'; metadata_path=fold_dir/\'complete.json\'\n            signature=stable_signature({\'config\':config,\'seed\':42,\'train\':train.adsh.tolist(),\n                \'inner_stop\':stop.adsh.tolist(),\'outer_score\':validation.adsh.tolist(),\n                \'proxy_sha256\':dataset_hash,\'text_sha256\':text_hash,\n                \'trainer_sha256\':trainer_hash,\'core_sha256\':core_hash,\n                \'runtime\':runtime_signature})\n            cached=None\n            if probability_path.is_file() and metadata_path.is_file():\n                metadata=json.loads(metadata_path.read_text(encoding=\'utf-8\'))\n                if metadata.get(\'signature\')==signature:\n                    candidate=np.load(probability_path)\n                    if candidate.shape==(len(validation),) and np.isfinite(candidate).all():\n                        cached=(candidate,int(metadata[\'iterations\']))\n            if cached is not None:\n                p,count=cached\n                print(f"Resume cached CV member {config[\'name\']} {fit_year}->{valid_year}.",flush=True)\n            else:\n                p,count,_=fit_model(config,train,validation,numeric,categorical,42,valid=stop)\n                atomic_npy(probability_path,p)\n                atomic_json(metadata_path,{\'signature\':signature,\'iterations\':int(count),\n                                           \'gpu\':torch.cuda.get_device_name(0),\n                                           \'outer_year\':valid_year,\'inner_stop_year\':fit_year})\n            preds.append(p); iterations.append(count)\n            fold_scores.append(float(average_precision_score(validation.y_amendment_proxy,p)))\n            print(f"{config[\'name\']} fit through {fit_year-1}, stop on {fit_year}, score on {valid_year}: AP={fold_scores[-1]:.4f}, iterations={count}",flush=True)\n        row={**config,\'cv_balanced_pr_auc\':float(np.mean(fold_scores)),\n             \'fold_scores\':fold_scores,\'iterations\':iterations,\n             \'inner_stop_years\':[year for year,_ in folds],\n             \'outer_score_years\':[year for _,year in folds],\n             \'final_iterations\':int(np.median(iterations))}\n        leaderboard.append(row); cv[config[\'name\']]=preds\n        print(row,flush=True)\n    def cv_score(names):\n        scores=[]\n        for i,(_,year) in enumerate(folds):\n            part=matched_cohort(selection_sample[selection_sample.filed_date.dt.year.eq(year)],707)\n            combined=np.mean([cv[n][i] for n in names],axis=0)\n            scores.append(average_precision_score(part.y_amendment_proxy,combined))\n        return float(np.mean(scores))\n    ranked=sorted(cv,key=lambda n:cv_score([n]),reverse=True)\n    candidates=[ranked[:1],ranked[:min(3,len(ranked))]]\n    selected=max(candidates,key=cv_score)\n    selection={\'members\':selected,\'cv_balanced_pr_auc\':cv_score(selected),\n               \'single_model_ranking\':{n:cv_score([n]) for n in ranked}}\n    print(\'Selected from forward CV:\',selection,flush=True)\n    calibration=matched_cohort(selection_sample[selection_sample.filed_date.dt.year.eq(2023)],43)\n    development_pool=sample[sample.filed_date.between(\'2024-10-01\',\'2025-09-30\')].copy()\n    development=development_pool\n    development=matched_cohort(development,44)\n    scores={}\n    for name in selected:\n        config=next(c for c in configs if c[\'name\']==name)\n        numeric,categorical=columns_for(sample,config[\'tabular\'])\n        train=selection_sample[selection_sample.filed_date.between(\'2014-01-01\',\'2021-12-31\')]\n        bags=[]\n        artifacts=root/\'evidence_v5\'/\'mda_models\'; artifacts.mkdir(parents=True,exist_ok=True)\n        for seed in core.SEEDS:\n            bag=balanced_train(train,seed)\n            predict_rows=pd.concat([calibration,development_pool])\n            fixed=next(r[\'final_iterations\'] for r in leaderboard if r[\'name\']==name)\n            model_base=artifacts/f\'{name}_{seed}\'\n            model_file=model_base.with_suffix(\'.pt\' if config[\'family\']==\'mlp\' else \'.json\')\n            vectorizer_file=Path(str(model_base)+\'.tfidf.joblib\')\n            if config[\'family\']==\'form_specialists\':\n                specialist_files=[specialist_model_base(model_base,form).with_suffix(\'.json\')\n                                  for form in (\'10-K\',\'10-Q\')]\n                model_complete=all(path.is_file() for path in specialist_files)\n            else:\n                model_complete=(model_file.is_file() and\n                                (config[\'family\']!=\'tfidf_xgb\' or vectorizer_file.is_file()))\n            member_dir=root/\'evidence_v5\'/\'mda_member_cache\'/name/str(seed)\n            probability_path=member_dir/\'probabilities.npy\'; metadata_path=member_dir/\'complete.json\'\n            signature=stable_signature({\'config\':config,\'seed\':seed,\'fixed\':fixed,\n                \'train\':bag.adsh.tolist(),\'predict\':predict_rows.adsh.tolist(),\n                \'proxy_sha256\':dataset_hash,\'text_sha256\':text_hash,\n                \'trainer_sha256\':trainer_hash,\'core_sha256\':core_hash,\n                \'runtime\':runtime_signature})\n            cached=None\n            if probability_path.is_file() and metadata_path.is_file() and model_complete:\n                metadata=json.loads(metadata_path.read_text(encoding=\'utf-8\'))\n                if metadata.get(\'signature\')==signature:\n                    candidate=np.load(probability_path)\n                    if candidate.shape==(len(predict_rows),) and np.isfinite(candidate).all():\n                        cached=(candidate,int(metadata[\'iterations\']))\n            if cached is not None:\n                p,count=cached\n                print(f\'Resume cached final member {name}, seed {seed}.\',flush=True)\n            else:\n                print(f\'Final GPU fit {name}, seed {seed}, {len(bag):,} case-control rows.\',flush=True)\n                p,count,_=fit_model(config,bag,predict_rows,numeric,categorical,seed,fixed=fixed,\n                                    save_path=model_base)\n                atomic_npy(probability_path,p)\n                atomic_json(metadata_path,{\'signature\':signature,\'iterations\':int(count),\n                                           \'gpu\':torch.cuda.get_device_name(0),\n                                           \'family\':config[\'family\'],\'train_rows\':len(bag)})\n            bags.append(p)\n        scores[name]=np.mean(bags,axis=0)\n    ensemble=np.mean([scores[name] for name in selected],axis=0)\n    calibration_scores=ensemble[:len(calibration)]\n    development_pool_scores=ensemble[len(calibration):]\n    score_lookup=pd.Series(development_pool_scores,index=development_pool.adsh)\n    development_scores=development.adsh.map(score_lookup).to_numpy(dtype=float)\n    threshold=core.threshold_choice(calibration.y_amendment_proxy,calibration_scores)\n    form_threshold_selection=calibrated_form_thresholds(\n        calibration,calibration_scores,threshold[\'threshold\'])\n    calibration_form_thresholds=thresholds_for_frame(\n        calibration,form_threshold_selection,threshold[\'threshold\'])\n    development_form_thresholds=thresholds_for_frame(\n        development,form_threshold_selection,threshold[\'threshold\'])\n    sample_sensitivity=[]\n    for seed in range(100,120):\n        cohort=matched_cohort(development_pool,seed)\n        cohort_scores=cohort.adsh.map(score_lookup).to_numpy(dtype=float)\n        sample_sensitivity.append({\'seed\':seed,**score_report(\n            cohort.y_amendment_proxy,cohort_scores,threshold[\'threshold\']),\n            \'form_calibrated\':score_report(cohort.y_amendment_proxy,cohort_scores,\n                thresholds_for_frame(cohort,form_threshold_selection,threshold[\'threshold\']))})\n    sensitivity_metrics=(\'accuracy\',\'precision\',\'recall\',\'f1\',\'pr_auc\',\'roc_auc\')\n    sensitivity_summary={name:{\'mean\':float(np.mean([r[name] for r in sample_sensitivity])),\n                               \'std\':float(np.std([r[name] for r in sample_sensitivity],ddof=1)),\n                               \'min\':float(np.min([r[name] for r in sample_sensitivity])),\n                               \'max\':float(np.max([r[name] for r in sample_sensitivity]))}\n                         for name in sensitivity_metrics}\n    form_sensitivity_summary={name:{\'mean\':float(np.mean([r[\'form_calibrated\'][name] for r in sample_sensitivity])),\n                                    \'std\':float(np.std([r[\'form_calibrated\'][name] for r in sample_sensitivity],ddof=1)),\n                                    \'min\':float(np.min([r[\'form_calibrated\'][name] for r in sample_sensitivity])),\n                                    \'max\':float(np.max([r[\'form_calibrated\'][name] for r in sample_sensitivity]))}\n                              for name in sensitivity_metrics}\n    issuer_development_pool=development_pool.loc[\n        issuer_holdout_mask(development_pool.cik)].copy()\n    issuer_development=None\n    if not issuer_development_pool.empty:\n        try:\n            issuer_development=matched_cohort(issuer_development_pool,44)\n            issuer_development_scores=issuer_development.adsh.map(score_lookup).to_numpy(dtype=float)\n            issuer_disjoint_metrics=score_report(issuer_development.y_amendment_proxy,\n                                                 issuer_development_scores,threshold[\'threshold\'])\n            issuer_disjoint_form_metrics=score_report(\n                issuer_development.y_amendment_proxy,issuer_development_scores,\n                thresholds_for_frame(issuer_development,form_threshold_selection,threshold[\'threshold\']))\n        except ValueError as exc:\n            issuer_disjoint_metrics={\'status\':\'insufficient_year_form_matches\',\'detail\':str(exc),\n                                     \'eligible_rows\':len(issuer_development_pool)}\n            issuer_disjoint_form_metrics={\'status\':\'insufficient_year_form_matches\',\'detail\':str(exc),\n                                          \'eligible_rows\':len(issuer_development_pool)}\n    else:\n        issuer_disjoint_metrics={\'status\':\'no_heldout_issuers_in_development_period\',\'eligible_rows\':0}\n        issuer_disjoint_form_metrics={\'status\':\'no_heldout_issuers_in_development_period\',\'eligible_rows\':0}\n    development_ci=issuer_cluster_confidence_intervals(\n        development,development_scores,threshold[\'threshold\'])\n    issuer_development_ci=(issuer_cluster_confidence_intervals(\n        issuer_development,issuer_development_scores,threshold[\'threshold\'],seed=2027)\n        if issuer_development is not None and \'n\' in issuer_disjoint_metrics else None)\n    development_form_ci=issuer_cluster_confidence_intervals(\n        development,development_scores,development_form_thresholds,seed=2028)\n    issuer_development_form_ci=(issuer_cluster_confidence_intervals(\n        issuer_development,issuer_development_scores,\n        thresholds_for_frame(issuer_development,form_threshold_selection,threshold[\'threshold\']),seed=2029)\n        if issuer_development is not None and \'n\' in issuer_disjoint_form_metrics else None)\n    metrics={\'evaluation_status\':\'Previously inspected development case-control period; issuer-disjoint subset is an exploratory cold-start check; no natural-prevalence metric is estimated\',\n        \'target\':\'Current FSDS same-period 365-day amendment proxy joined by accession; old corpus y excluded\',\n        \'sampling_warning\':\'The MD&A corpus is an outcome-selected, approximately balanced sample. These results apply only to the matched text-bearing case-control cohort.\',\n        \'gpu\':gpu_name,\'dataset_sha256\':dataset_hash,\n        \'text_dataset_sha256\':text_hash,\n        \'code_sha256\':{\'trainer\':core.file_hash(__file__),\'core\':core.file_hash(core.__file__)},\n        \'runtime_versions\':runtime_versions,\n        \'text_rows\':len(text),\'text_available_rows\':int(text.text_available.sum()),\n        \'text_status_counts\':(text.text_status.fillna(\'unknown\').value_counts().to_dict()\n                              if \'text_status\' in text else {\'provided_corpus\':len(text)}),\n        \'text_positive\':int(text.y_amendment_proxy.sum()),\n        \'full_proxy_rows\':proxy_row_count,\'full_proxy_positive_count\':proxy_positive_count,\n        \'legacy_y_used\':False,\n        \'selection\':selection,\'cv_leaderboard\':leaderboard,\n        \'calibration\':score_report(calibration.y_amendment_proxy,calibration_scores,threshold[\'threshold\']),\n        \'calibration_form_specific_thresholds\':{\n            \'thresholds\':form_threshold_selection,\n            \'metrics\':score_report(calibration.y_amendment_proxy,calibration_scores,\n                                   calibration_form_thresholds),\n            \'rule\':\'Each 10-K/10-Q threshold is calibrated on 2023 only; groups below 30 examples per class use the global calibration threshold.\'},\n        \'development\':score_report(development.y_amendment_proxy,development_scores,threshold[\'threshold\']),\n        \'development_form_specific_thresholds\':score_report(\n            development.y_amendment_proxy,development_scores,development_form_thresholds),\n        \'development_issuer_cluster_ci\':development_ci,\n        \'development_form_threshold_issuer_cluster_ci\':development_form_ci,\n        \'issuer_disjoint_development\':issuer_disjoint_metrics,\n        \'issuer_disjoint_form_specific_thresholds\':issuer_disjoint_form_metrics,\n        \'issuer_disjoint_issuer_cluster_ci\':issuer_development_ci,\n        \'issuer_disjoint_form_threshold_issuer_cluster_ci\':issuer_development_form_ci,\n        \'issuer_holdout\':{\'salt\':\'mda-expanded-cik-v1\',\'fraction\':.20,\n                          \'heldout_ciks\':int(sample.loc[heldout_issuer,\'cik\'].nunique()),\n                          \'selection_ciks\':int(sample.loc[~heldout_issuer,\'cik\'].nunique()),\n                          \'selection_accessions\':len(selection_sample),\n                          \'interpretation\':\'Model selection and fitting exclude these CIKs. The period overlaps previously inspected development dates, so this is not an untouched prospective test.\'},\n        \'development_control_sampling_sensitivity\':{\n            \'description\':\'Twenty year-form matched 1:1 draws from the same text-bearing development pool; the calibrated threshold is kept fixed.\',\n            \'draws\':sample_sensitivity,\'summary\':sensitivity_summary,\n            \'form_specific_threshold_summary\':form_sensitivity_summary},\n        \'threshold_selection\':threshold,\n        \'development_rows_before_matching\':int(sample.filed_date.between(\'2024-10-01\',\'2025-09-30\').sum()),\n        \'development_text_rows_after_matching\':len(development),\n        \'text_only_encoder\':\'sentence-transformers/all-MiniLM-L6-v2; frozen; GPU embeddings; beginning/middle/ending means and chunk max pooling\',\n        \'text_style_features\':\'Word count, lexical diversity, token/sentence length, numeric density, and transparent generic language-group rates; computed from supplied MD&A text\',\n        \'text_change_features\':\'HashingVectorizer word unigram/bigram similarity and novelty plus embedding deltas from the strictly earlier same-issuer/form filing when both texts are available\',\n        \'new_model_families\':[\'GPU pairwise XGBoost ranking by year/form query\',\'GPU XGBoost over train-fold-only word/character TF-IDF plus structured features\'],\n        \'encoder_revision\':revision,\'encoder_license\':\'Apache-2.0\',\n        \'cv_interpretation\':\'The chronological folds and leaderboard are used for model/ensemble selection; their scores are not an untouched estimate after trying multiple candidates.\',\n        \'tabular_reference\':\'The matching-cohort v5 scores are saved from the existing v5 development predictions when available.\'}\n    # Compare the v5 balanced-score threshold on the exact same matched rows.\n    prior_candidates=list(root.glob(\'colab_outputs/research_v5_*/development_predictions.csv.gz\'))\n    prior_candidates+=list(root.glob(\'research_v5_*/development_predictions.csv.gz\'))\n    prior_candidates+=list(root.glob(\'research_v5_*/research_v5_*/development_predictions.csv.gz\'))\n    prior=sorted(prior_candidates,key=lambda p:p.stat().st_mtime,reverse=True)\n    if prior:\n        previous=pd.read_csv(prior[0],compression=\'gzip\',usecols=[\'adsh\',\'balanced_score\'])\n        common=development[[\'adsh\',\'y_amendment_proxy\']].merge(previous,on=\'adsh\',how=\'inner\',validate=\'one_to_one\')\n        if len(common)==len(development):\n            previous_metrics=json.loads((prior[0].parent/\'metrics.json\').read_text(encoding=\'utf-8\'))\n            old_threshold=previous_metrics[\'balanced\'][\'threshold_selection\'][\'threshold\']\n            metrics[\'v5_tabular_on_same_matched_development\']=score_report(\n                common.y_amendment_proxy,common.balanced_score,old_threshold)\n            metrics[\'v5_tabular_threshold_note\']=\'The prior V5 calibration threshold is applied unchanged; this is a selected case-control text cohort, not the full population.\'\n    out=root/\'colab_outputs\'/f\'{experiment_name}_{hashlib.sha256((dataset_hash+text_hash+trainer_hash).encode()).hexdigest()[:12]}\'\n    out.mkdir(parents=True,exist_ok=True)\n    atomic_json(out/\'metrics.json\',metrics)\n    prediction=development[[\'adsh\',\'cik\',\'form\',\'filed_date\',\'y_amendment_proxy\']].copy()\n    prediction[\'text_ensemble_score\']=development_scores\n    prediction[\'text_ensemble_prediction\']=(development_scores>=threshold[\'threshold\']).astype(\'int8\')\n    prediction[\'form_calibrated_threshold\']=development_form_thresholds\n    prediction[\'form_calibrated_prediction\']=(development_scores>=development_form_thresholds).astype(\'int8\')\n    prediction.to_csv(out/\'development_predictions_case_control.csv.gz\',index=False,compression=\'gzip\')\n    if issuer_development is not None and \'n\' in issuer_disjoint_metrics:\n        issuer_prediction=issuer_development[[\'adsh\',\'cik\',\'form\',\'filed_date\',\'y_amendment_proxy\']].copy()\n        issuer_prediction[\'text_ensemble_score\']=issuer_development_scores\n        issuer_prediction[\'text_ensemble_prediction\']=(issuer_development_scores>=threshold[\'threshold\']).astype(\'int8\')\n        issuer_form_thresholds=thresholds_for_frame(\n            issuer_development,form_threshold_selection,threshold[\'threshold\'])\n        issuer_prediction[\'form_calibrated_threshold\']=issuer_form_thresholds\n        issuer_prediction[\'form_calibrated_prediction\']=(\n            issuer_development_scores>=issuer_form_thresholds).astype(\'int8\')\n        issuer_prediction.to_csv(out/\'issuer_disjoint_predictions_case_control.csv.gz\',index=False,compression=\'gzip\')\n    pd.DataFrame(leaderboard).to_csv(out/\'cv_leaderboard.csv\',index=False)\n    print(json.dumps({\'calibration\':metrics[\'calibration\'],\'development\':metrics[\'development\'],\n                      \'same_cohort_v5\':metrics.get(\'v5_tabular_on_same_matched_development\'),\n                      \'output\':str(out)},indent=2),flush=True)\n    return out\n'}
for name,source in SOURCES.items(): (MODULES/name).write_text(source,encoding="utf-8")
if str(MODULES) not in sys.path: sys.path.insert(0,str(MODULES))
import research_v5_core as core
importlib.reload(core)
print("Embedded research code loaded.")

In [ ]:
spec=importlib.util.spec_from_file_location("mda_training",MODULES/"17_train_mda_text_gpu.py")
training=importlib.util.module_from_spec(spec); spec.loader.exec_module(training)
OUTPUT=training.run(ROOT,text_path=TEXT_PATH,experiment_name="mda_text_expanded")
print("Output folder:",OUTPUT)

In [ ]:
import json,pandas as pd
report=json.loads((OUTPUT/"metrics.json").read_text())
display(pd.DataFrame({"development":report["development"],
                     "issuer-disjoint development":report["issuer_disjoint_development"]}).T)
display(pd.DataFrame(report["development_control_sampling_sensitivity"]["summary"]).T)
print("Return metrics.json, cv_leaderboard.csv, development_predictions_case_control.csv.gz, and issuer_disjoint_predictions_case_control.csv.gz from",OUTPUT)